# Pipeline scRNA-seq: scanpy + SCENIC

### Un solo notebook, dos modos de datos — elige uno y ejecuta todo

Este notebook corre el pipeline completo (**clustering** + **redes regulatorias SCENIC**) sobre uno de dos conjuntos de datos, según lo que elijas en **una sola celda** al inicio:

| Modo | Datos | Qué hace de principio a fin |
|---|---|---|
| **`"ejemplo"`** | Datos de demostración | Clustering de médula ósea humana + SCENIC sobre el dataset `tiny` |
| **`"zenodo"`** | Datos reales de lupus | Convierte los datos de Jang *et al.*, explora SLE vs control, reproduce las figuras + SCENIC sobre células B |

**Cómo usarlo:**
1. Ve a la celda **1 · Configuración** y pon `MODO = "ejemplo"` o `MODO = "zenodo"`.
2. Menú *Entorno de ejecución → Ejecutar todas*. El notebook corre entero para ese modo.
3. Las celdas del otro modo se **saltan solas** (no dan error).

> Para el modo `"zenodo"` conviene **RAM alta** si vas a usar todas las células; con el downsample por defecto funciona en Colab gratis.

**Referencias:** scanpy · SCENICprotocol (aertslab) · Jang *et al.*, Arthritis & Rheumatology 2025 (doi 10.1002/art.70116) · datos: zenodo.org/records/17868028


---
# 1 · Configuración

**Esta es la única celda que tienes que editar.** Elige el modo y, si usas `"zenodo"`, ajusta los parámetros de memoria y de SCENIC.


In [ ]:
# ============================================================
#   CELDA MAESTRA  —  elige aqui que quieres correr
# ============================================================

MODO = "ejemplo"          # "ejemplo"  o  "zenodo"

# ---- Parametros solo para MODO = "zenodo" ------------------
N_CELLS_MAX       = 60000  # celulas a conservar al convertir el RDS (0 = todas; requiere RAM alta)

# ---- Parametros de SCENIC (Parte B) ------------------------
SCENIC_DOWNSAMPLE = True    # True = recorta para que corra rapido; False = usa TODAS las celulas B (lento, horas)
SCENIC_N_CELLS    = 2000    # nº de celulas B para SCENIC si DOWNSAMPLE=True
SCENIC_N_GENES    = 1500    # nº de genes variables (HVG) a usar como objetivo en GRNBoost2
# ============================================================

assert MODO in ("ejemplo", "zenodo"), "MODO debe ser 'ejemplo' o 'zenodo'"
print(f"MODO seleccionado: {MODO}")
if MODO == "zenodo":
    print(f"  Downsample conversion : {N_CELLS_MAX if N_CELLS_MAX else 'TODAS'} celulas")
    print(f"  SCENIC downsample     : {'SI ('+str(SCENIC_N_CELLS)+' celulas B)' if SCENIC_DOWNSAMPLE else 'NO (todas)'}")


---
# 2 · Preparar el entorno

Instalamos las librerías (común a los dos modos). Tarda 2-4 minutos. Si Colab pide *reiniciar entorno*, hazlo y vuelve a *Ejecutar todas*.


### 2.1 · Revisar memoria disponible

In [ ]:
import psutil, os
ram_gb = psutil.virtual_memory().total / 1e9
print(f"RAM total: {ram_gb:.1f} GB | CPUs: {os.cpu_count()}")
if MODO == "zenodo" and ram_gb < 20 and N_CELLS_MAX == 0:
    print("AVISO: pediste TODAS las celulas (N_CELLS_MAX=0) pero hay poca RAM.")
    print("       Activa RAM alta o usa N_CELLS_MAX=60000.")


### 2.2 · Instalar scanpy (clustering)

In [ ]:
!pip install -q scanpy leidenalg igraph scikit-misc 2>/dev/null
print("scanpy listo")


### 2.3 · Instalar pySCENIC (Parte B)

Ambos modos usan SCENIC, así que instalamos pySCENIC siempre. Es la instalación más delicada; si falla, reinicia el entorno y re-ejecuta solo esta celda.


In [ ]:
!pip install -q pyscenic==0.12.1 2>/dev/null
try:
    import pyscenic, ctxcore, loompy
    print("pyscenic:", pyscenic.__version__, "| loompy:", loompy.__version__)
except Exception as e:
    print("Si hay error, reinicia el entorno y re-ejecuta esta celda.\n", e)


### 2.4 · (Opcional) Conectar Google Drive para guardar resultados

In [ ]:
try:
    from google.colab import drive
    drive.mount('/content/drive')
    PROJECT_DIR = '/content/drive/MyDrive/scanpy_scenic_lupus'
    os.makedirs(PROJECT_DIR, exist_ok=True)
    print("Resultados en:", PROJECT_DIR)
except Exception as e:
    PROJECT_DIR = '.'
    print("Sin Drive; se guarda en la sesion temporal.")


---
# 3 · Parte 1 — Datos y clustering

A partir de aquí el notebook se bifurca según `MODO`. Cada bloque empieza con un aviso de a qué modo pertenece; las celdas del otro modo se saltan solas.

El objetivo de esta parte es producir un objeto **`adata`** (la estructura de datos de scanpy) analizado y listo. En modo `ejemplo` se construye desde datos crudos; en modo `zenodo` se convierte desde el archivo de R.


## 3A · [MODO EJEMPLO] Clustering de médula ósea

> Estas celdas **solo corren si `MODO == "ejemplo"`**. Reproducen el tutorial de scanpy: médula ósea humana (~17.000 células) desde datos crudos hasta tipos celulares.


### 3A.1 · Cargar los datos de ejemplo (10X Genomics)

In [ ]:
if MODO == "ejemplo":
    import scanpy as sc, anndata as ad, numpy as np, pandas as pd, pooch
    sc.settings.verbosity = 1
    sc.settings.set_figure_params(dpi=70, facecolor='white')
    np.random.seed(0)

    EX = pooch.create(path=pooch.os_cache('scverse_tutorials'),
                      base_url='doi:10.6084/m9.figshare.22716739.v1/')
    EX.load_registry_from_doi()
    samples = {'s1d1':'s1d1_filtered_feature_bc_matrix.h5',
               's1d3':'s1d3_filtered_feature_bc_matrix.h5'}
    adatas = {}
    for sid, fn in samples.items():
        a = sc.read_10x_h5(EX.fetch(fn)); a.var_names_make_unique()
        adatas[sid] = a
        print(f"{sid}: {a.n_obs:,} x {a.n_vars:,}")
    adata = ad.concat(adatas, label='sample'); adata.obs_names_make_unique()
    print("Combinado:", adata.shape)


### 3A.2 · Control de calidad (mitocondrial, ribosomal, hemoglobina)

In [ ]:
if MODO == "ejemplo":
    adata.var['mt']   = adata.var_names.str.startswith('MT-')
    adata.var['ribo'] = adata.var_names.str.startswith(('RPS','RPL'))
    adata.var['hb']   = adata.var_names.str.contains('^HB[^(P)]')
    sc.pp.calculate_qc_metrics(adata, qc_vars=['mt','ribo','hb'], inplace=True, log1p=True)
    sc.pl.violin(adata, ['n_genes_by_counts','total_counts','pct_counts_mt'],
                 jitter=0.4, multi_panel=True)
    n0 = adata.n_obs
    sc.pp.filter_cells(adata, min_genes=100)
    sc.pp.filter_genes(adata, min_cells=3)
    print(f"Celulas: {n0:,} -> {adata.n_obs:,} | genes: {adata.n_vars:,}")


### 3A.3 · Dobletes, normalización y genes variables

In [ ]:
if MODO == "ejemplo":
    sc.pp.scrublet(adata, batch_key='sample', random_state=0)
    print(f"Dobletes: {adata.obs['predicted_doublet'].sum():,}")
    adata.layers['counts'] = adata.X.copy()
    sc.pp.normalize_total(adata); sc.pp.log1p(adata)
    sc.pp.highly_variable_genes(adata, n_top_genes=2000, batch_key='sample')
    print(f"HVG: {adata.var.highly_variable.sum():,}")


### 3A.4 · PCA, vecinos, UMAP y clustering Leiden

In [ ]:
if MODO == "ejemplo":
    sc.tl.pca(adata, random_state=0)
    sc.pp.neighbors(adata, random_state=0)
    sc.tl.umap(adata, random_state=0)
    for res in [0.02, 0.5, 2.0]:
        sc.tl.leiden(adata, key_added=f'leiden_res_{res:4.2f}', resolution=res,
                     flavor='igraph', n_iterations=2, random_state=0)
        print(f"res {res}: {adata.obs[f'leiden_res_{res:4.2f}'].nunique()} clusters")
    sc.pl.umap(adata, color=['sample','leiden_res_0.02','leiden_res_0.50'], wspace=0.3, ncols=3)


### 3A.5 · Anotación de tipos celulares con marcadores

In [ ]:
if MODO == "ejemplo":
    marker_genes = {
        'CD14+ Mono': ['FCN1','CD14'], 'CD16+ Mono': ['TCF7L2','FCGR3A','LYN'],
        'cDC2': ['CST3','COTL1','LYZ','CLEC10A','FCER1A'],
        'Erythroblast': ['MKI67','HBA1','HBB'], 'Proerythroblast': ['CDK6','SYNGR1','HBM','GYPA'],
        'NK': ['GNLY','NKG7','CD247','TYROBP','KLRG1'],
        'Naive CD20+ B': ['MS4A1','IL4R','IGHD','FCRL1','IGHM'],
        'Plasma cells': ['MZB1','HSP90B1','PRDM1','IGKC','JCHAIN'],
        'CD4+ T': ['CD4','IL7R','TRBC2'], 'CD8+ T': ['CD8A','CD8B','GZMK','CCL5','GZMB'],
        'T naive': ['LEF1','CCR7','TCF7'], 'pDC': ['IL3RA','COBLL1','TCF4'],
    }
    sc.pl.dotplot(adata, marker_genes, groupby='leiden_res_0.02', standard_scale='var')
    cluster_to_celltype = {'0':'Lymphocytes','1':'Monocytes','2':'Erythroid','3':'B Cells'}
    adata.obs['cell_type'] = (adata.obs['leiden_res_0.02'].map(cluster_to_celltype)
                              .fillna('Unknown').astype('category'))
    sc.pl.umap(adata, color='cell_type', legend_loc='on data')


### 3A.6 · Expresión diferencial y guardado

In [ ]:
if MODO == "ejemplo":
    sc.tl.rank_genes_groups(adata, groupby='leiden_res_0.50', method='wilcoxon')
    sc.pl.rank_genes_groups_dotplot(adata, groupby='leiden_res_0.50', standard_scale='var', n_genes=5)
    out = f"{PROJECT_DIR}/adata_medula_clustered.h5ad"
    adata.write_h5ad(out); print("Guardado:", out)


## 3B · [MODO ZENODO] Datos reales de lupus + rituximab

> Estas celdas **solo corren si `MODO == "zenodo"`**. Descargan y convierten el dataset de Jang *et al.*, exploran los grupos y reproducen las dos figuras del artículo.

**Estrategia de RAM:** la conversión de R corre en un **proceso separado** (`Rscript`), que libera toda su memoria al terminar, antes de que Python cargue nada. Más un downsample opcional (`N_CELLS_MAX`).


### 3B.1 · Descargar el objeto de Zenodo (1.6 GB)

In [ ]:
if MODO == "zenodo":
    import os
    RDS_PATH = 'RTX_zenodo.RDS'
    if not os.path.exists(RDS_PATH):
        print("Descargando RTX_zenodo.RDS (1.6 GB)...")
        !wget -q --show-progress -O {RDS_PATH} "https://zenodo.org/records/17868028/files/RTX_zenodo.RDS?download=1"
    print(f"Tamano: {os.path.getsize(RDS_PATH)/1e9:.2f} GB")


### 3B.2 · Instalar SeuratObject en R (para abrir el .RDS)

In [ ]:
if MODO == "zenodo":
    !Rscript -e 'if(!requireNamespace("SeuratObject",quietly=TRUE))install.packages("SeuratObject",repos="https://cloud.r-project.org"); if(!requireNamespace("Matrix",quietly=TRUE))install.packages("Matrix",repos="https://cloud.r-project.org"); cat("SeuratObject listo\n")'
else:
    print("[omitido] R solo se usa en modo zenodo")


### 3B.3 · Escribir el script de conversión de R

Solo escribe el archivo `convert_rds.R` en disco (inofensivo en modo ejemplo). Cuando se ejecute, cargará el RDS, **imprimirá los grupos y columnas**, aplicará el downsample y extraerá los datos a archivos simples.


In [ ]:
%%writefile convert_rds.R
suppressMessages({library(SeuratObject); library(Matrix)})
n_max <- as.integer(Sys.getenv("N_CELLS_MAX", "60000"))
obj <- readRDS("RTX_zenodo.RDS")

cat("=== ESTRUCTURA ===\n"); print(obj)
cat("\n=== ASSAYS ===\n"); print(Assays(obj))
cat("\n=== REDUCCIONES ===\n"); print(Reductions(obj))
cat("\n=== COLUMNAS METADATA ===\n"); print(colnames(obj@meta.data))
meta <- obj@meta.data
cat("\n=== GRUPOS (categoricas) ===\n")
for (col in colnames(meta)) {
    v <- meta[[col]]
    if (is.factor(v)||is.character(v)||(is.numeric(v)&&length(unique(v))<30)) {
        if (length(unique(v))<=30){cat("\n[",col,"]\n",sep="");print(table(v))}
    }
}
ncells <- ncol(obj)
if (n_max>0 && n_max<ncells) {
    set.seed(0)
    cc <- grep("celltype|cell_type|cell.type|annotation|ident",colnames(meta),ignore.case=TRUE,value=TRUE)
    if (length(cc)>0) {
        grp <- as.character(meta[[cc[1]]]); frac <- n_max/ncells
        idx <- sort(unlist(lapply(split(seq_len(ncells),grp),
                    function(ix) sample(ix, max(1, round(length(ix)*frac))))))
    } else idx <- sort(sample(ncells, n_max))
    cat("\n>>> DOWNSAMPLE:", ncells, "->", length(idx), "celulas\n")
} else { idx <- seq_len(ncells); cat("\n>>> TODAS:", ncells, "celulas\n") }

assay_rna <- if ("RNA" %in% Assays(obj)) "RNA" else DefaultAssay(obj)
counts <- tryCatch(GetAssayData(obj,assay=assay_rna,slot="counts"),
                   error=function(e) GetAssayData(obj,assay=assay_rna,layer="counts"))
counts <- counts[, idx]
Matrix::writeMM(counts, "counts.mtx")
write.csv(data.frame(gene=rownames(counts)), "genes.csv", row.names=FALSE)
write.csv(data.frame(barcode=colnames(counts)), "barcodes.csv", row.names=FALSE)
write.csv(meta[idx,,drop=FALSE], "metadata.csv")
reds <- Reductions(obj); un <- reds[grepl("umap",tolower(reds))][1]
if (is.na(un)) un <- reds[1]
write.csv(Embeddings(obj,un)[idx,,drop=FALSE], "umap.csv")
rm(obj,counts,meta); gc()
cat("\nEmbedding:", un, "\nExtraccion completa. RAM de R liberada.\n")


### 3B.4 · Ejecutar la conversión

Corre el script en un proceso de R separado. **Lee el output**: te muestra los grupos y columnas (SLE/control, timepoints, tipos celulares) — los necesitas para ajustar las celdas de figuras.


In [ ]:
if MODO == "zenodo":
    import os
    os.environ['N_CELLS_MAX'] = str(N_CELLS_MAX)
    !N_CELLS_MAX=$N_CELLS_MAX Rscript convert_rds.R
    print("\nArchivos: counts.mtx, genes.csv, barcodes.csv, metadata.csv, umap.csv")


### 3B.5 · Reconstruir `adata` en Python (con liberación de RAM)

In [ ]:
if MODO == "zenodo":
    import scipy.io, gc, scanpy as sc, anndata as ad, pandas as pd, numpy as np
    sc.settings.set_figure_params(dpi=70, facecolor='white')
    X = scipy.io.mmread('counts.mtx').T.tocsr()
    genes = pd.read_csv('genes.csv')['gene'].astype(str).values
    bc    = pd.read_csv('barcodes.csv')['barcode'].astype(str).values
    meta  = pd.read_csv('metadata.csv', index_col=0)
    umap  = pd.read_csv('umap.csv', index_col=0)
    adata = ad.AnnData(X=X, obs=meta, var=pd.DataFrame(index=genes))
    adata.obs_names = bc
    adata.obsm['X_umap'] = umap.values
    del X, meta, umap; gc.collect()
    print(adata)
    print("\nColumnas:", list(adata.obs.columns))


### 3B.6 · Guardar h5ad y liberar disco

In [ ]:
if MODO == "zenodo":
    import os, gc
    out = f"{PROJECT_DIR}/lupus_rituximab.h5ad"
    adata.write_h5ad(out); print("Guardado:", out)
    if os.path.exists('counts.mtx'): os.remove('counts.mtx')
    gc.collect()


### 3B.7 · Detectar las columnas de grupos (condición, timepoint, tipo celular)

In [ ]:
if MODO == "zenodo":
    obs = adata.obs
    def find_col(cs):
        for c in obs.columns:
            if any(k in c.lower() for k in cs): return c
        return None
    col_cond  = find_col(['disease','condition','group','sle','status','diagnosis'])
    col_time  = find_col(['time','visit','day','week','treatment','point'])
    col_ctype = find_col(['celltype','cell_type','cell.type','annotation','ident','cluster'])
    print("condicion :", col_cond)
    print("timepoint :", col_time)
    print("tipo cel. :", col_ctype)
    for c in [col_cond, col_time, col_ctype]:
        if c: print(f"\n[{c}]\n", obs[c].value_counts())


### 3B.8 · UMAP por grupo y tipo celular

In [ ]:
if MODO == "zenodo":
    cols = [c for c in [col_ctype, col_cond, col_time] if c]
    sc.pl.umap(adata, color=cols, wspace=0.4, ncols=2, size=3)


### 3B.9 · Figura 1 — UMAP de subtipos de células B

Filtra a las células B y las colorea por subtipo (Naive, Transitional, Memory, Switched, ABCs, Plasmablasts, Plasma). Ajusta `col_ctype` si el nombre difiere.


In [ ]:
if MODO == "zenodo":
    B_KW = ['naive b','transitional','memory b','switched','abc','plasmablast','plasma','b cell','b_cell']
    if col_ctype:
        isB = adata.obs[col_ctype].astype(str).str.lower().str.contains('|'.join(B_KW))
        adata_B = adata[isB].copy()
        print(f"Celulas B: {adata_B.n_obs:,}")
        print(adata_B.obs[col_ctype].value_counts())
        sc.pl.umap(adata_B, color=col_ctype, size=8,
                   title='Subtipos de celulas B (Lupus + Rituximab)')
    else:
        print("Ajusta col_ctype manualmente.")


### 3B.10 · Figura 2 — Volcano plot pre vs post-rituximab

Compara la expresión antes vs temprano-después del tratamiento. **Ajusta `PRE_LABEL` y `POST_LABEL`** con los valores reales de tu columna de timepoint (celda 3B.7).


In [ ]:
if MODO == "zenodo":
    PRE_LABEL  = 'Pretreatment'          # <-- AJUSTAR
    POST_LABEL = 'Early post-treatment'  # <-- AJUSTAR
    CELLTYPE_SUBSET = None               # p.ej. 'CD4 T Naive' o None

    ad_de = adata
    if CELLTYPE_SUBSET and col_ctype:
        ad_de = adata[adata.obs[col_ctype].astype(str) == CELLTYPE_SUBSET]
    ad_de = ad_de.copy()
    if ad_de.X.max() > 50:
        sc.pp.normalize_total(ad_de, target_sum=1e4); sc.pp.log1p(ad_de)
    mask = ad_de.obs[col_time].astype(str).isin([PRE_LABEL, POST_LABEL])
    ad_de = ad_de[mask].copy()
    print(ad_de.obs[col_time].value_counts())
    sc.tl.rank_genes_groups(ad_de, groupby=col_time, groups=[POST_LABEL],
                            reference=PRE_LABEL, method='wilcoxon')
    de = sc.get.rank_genes_groups_df(ad_de, group=POST_LABEL)


In [ ]:
if MODO == "zenodo":
    import matplotlib.pyplot as plt, numpy as np
    d = de.dropna(subset=['logfoldchanges','pvals_adj']).copy()
    d['nlp'] = -np.log10(d['pvals_adj'].clip(lower=1e-300))
    up   = (d['logfoldchanges']>1) & (d['pvals_adj']<0.05)
    down = (d['logfoldchanges']<-1) & (d['pvals_adj']<0.05)
    plt.figure(figsize=(8,6))
    plt.scatter(d['logfoldchanges'], d['nlp'], s=6, c='lightgray')
    plt.scatter(d.loc[up,'logfoldchanges'], d.loc[up,'nlp'], s=8, c='#c0392b', label='Up (post)')
    plt.scatter(d.loc[down,'logfoldchanges'], d.loc[down,'nlp'], s=8, c='#2e5f9a', label='Down (post)')
    for _, r in d[up|down].nlargest(15,'nlp').iterrows():
        plt.text(r['logfoldchanges'], r['nlp'], r['names'], fontsize=7)
    plt.axvline(0,color='k',lw=.5); plt.axhline(-np.log10(0.05),color='k',ls='--',lw=.5)
    plt.xlabel(f'logFC ({POST_LABEL} - {PRE_LABEL})'); plt.ylabel('-log10 (p ajustado)')
    plt.title('Volcano: post vs pre-rituximab'); plt.legend(markerscale=2, fontsize=8)
    plt.tight_layout(); plt.show()


---
# 4 · Parte 2 — Redes regulatorias (SCENIC)

SCENIC descubre qué **factores de transcripción** controlan a qué genes (regulones). Corre en **ambos modos**, pero sobre datos distintos:

| Modo | SCENIC corre sobre |
|---|---|
| `ejemplo` | dataset de prueba `tiny` (rápido) |
| `zenodo` | **células B reales del lupus** (con downsample según `SCENIC_DOWNSAMPLE`) |

Flujo idéntico en ambos: **GRNBoost2** (sklearn) → **cisTarget** → **AUCell**.


### 4.1 · Descargar bases de datos de SCENIC

Motivos y ranking genómico hg38 (~410 MB) se usan en los dos modos. La lista de TFs cambia según el modo.


In [ ]:
import os, urllib.request
os.makedirs('scenic_data', exist_ok=True)
COM = {
  'motifs.tbl':'https://resources.aertslab.org/cistarget/motif2tf/motifs-v10nr_clust-nr.hgnc-m0.001-o0.0.tbl',
  'hg38_rankings.feather':'https://resources.aertslab.org/cistarget/databases/homo_sapiens/hg38/refseq_r80/mc_v10_clust/gene_based/hg38_10kbp_up_10kbp_down_full_tx_v10_clust.genes_vs_motifs.rankings.feather',
}
if MODO == "ejemplo":
    COM['expr_mat_tiny.loom'] = 'https://raw.githubusercontent.com/aertslab/SCENICprotocol/master/example/expr_mat_tiny.loom'
    COM['test_TFs_tiny.txt']  = 'https://raw.githubusercontent.com/aertslab/SCENICprotocol/master/example/test_TFs_tiny.txt'
else:
    COM['allTFs_hg38.txt'] = 'https://resources.aertslab.org/cistarget/tf_lists/allTFs_hg38.txt'

for fn, url in COM.items():
    dest = f'scenic_data/{fn}'
    if not os.path.exists(dest):
        print("Descargando", fn, "...")
        urllib.request.urlretrieve(url, dest)
    print(f"  {fn}: {os.path.getsize(dest)/1e6:.1f} MB")


### 4.2 · Preparar la matriz de expresión y la lista de TFs

Aquí es donde el modo decide los datos de entrada de SCENIC. El resto del pipeline (4.3-4.6) es idéntico.


In [ ]:
import pandas as pd, numpy as np

if MODO == "ejemplo":
    import loompy
    with loompy.connect('scenic_data/expr_mat_tiny.loom') as ds:
        ex_matrix = pd.DataFrame(ds[:,:].T, index=ds.ca['CellID'], columns=ds.ra['Gene'])
    tf_names = pd.read_csv('scenic_data/test_TFs_tiny.txt', header=None).iloc[:,0].tolist()
    N_EST = 500

else:
    import scanpy as sc
    # Partimos de las celulas B (de la Figura 1). Si no existe adata_B, lo creamos.
    if 'adata_B' not in dir():
        B_KW = ['naive b','transitional','memory b','switched','abc','plasmablast','plasma','b cell','b_cell']
        isB = adata.obs[col_ctype].astype(str).str.lower().str.contains('|'.join(B_KW))
        adata_B = adata[isB].copy()
    ad_s = adata_B.copy()

    # Downsample de celulas (opcional)
    if SCENIC_DOWNSAMPLE and ad_s.n_obs > SCENIC_N_CELLS:
        sc.pp.subsample(ad_s, n_obs=SCENIC_N_CELLS, random_state=0)
    print(f"Celulas B para SCENIC: {ad_s.n_obs:,}")

    # Normalizar y quedarnos con los genes mas variables (reduce el coste)
    if ad_s.X.max() > 50:
        sc.pp.normalize_total(ad_s, target_sum=1e4); sc.pp.log1p(ad_s)
    sc.pp.highly_variable_genes(ad_s, n_top_genes=min(SCENIC_N_GENES, ad_s.n_vars-1))
    ad_s = ad_s[:, ad_s.var.highly_variable].copy()

    Xd = ad_s.X.toarray() if hasattr(ad_s.X,'toarray') else np.asarray(ad_s.X)
    ex_matrix = pd.DataFrame(Xd, index=ad_s.obs_names.astype(str), columns=ad_s.var_names.astype(str))
    all_tfs = pd.read_csv('scenic_data/allTFs_hg38.txt', header=None).iloc[:,0].tolist()
    tf_names = [t for t in all_tfs if t in ex_matrix.columns]
    N_EST = 200  # menos estimadores: datos reales son mas pesados
    if not SCENIC_DOWNSAMPLE:
        print("AVISO: SCENIC_DOWNSAMPLE=False -> puede tardar HORAS.")

print(f"Matriz: {ex_matrix.shape[0]} celulas x {ex_matrix.shape[1]} genes | TFs: {len(tf_names)}")


### 4.3 · GRNBoost2 con scikit-learn

Para cada gen, regresa su expresión contra los TFs y extrae las importancias. (Reemplaza a `arboreto`, incompatible con dask moderno.)


In [ ]:
from sklearn.ensemble import GradientBoostingRegressor

tfs_in = [t for t in tf_names if t in ex_matrix.columns]
X_tfs  = ex_matrix[tfs_in].values
targets = [g for g in ex_matrix.columns if g not in tfs_in]
print(f"TFs en matriz: {len(tfs_in)} | genes objetivo: {len(targets)}")

records = []
for i, target in enumerate(targets):
    y = ex_matrix[target].values
    if y.std() == 0: continue
    gbm = GradientBoostingRegressor(n_estimators=N_EST, max_depth=3, random_state=42)
    gbm.fit(X_tfs, y)
    for tf, imp in zip(tfs_in, gbm.feature_importances_):
        if imp > 0: records.append({'TF':tf,'target':target,'importance':imp})
    if (i+1) % 200 == 0: print(f"  {i+1}/{len(targets)} genes procesados")

adjacencies = pd.DataFrame(records).sort_values('importance', ascending=False)
adjacencies.to_csv('scenic_data/adjacencies.tsv', sep='\t', index=False)
print(f"Relaciones TF-gen: {len(adjacencies):,}")
adjacencies.head()


### 4.4 · cisTarget — validación por motivos de ADN

Necesita la matriz de expresión en formato `.loom`. En modo zenodo la creamos desde la matriz de células B.


In [ ]:
import loompy, numpy as np
LOOM = 'scenic_data/expr_mat_tiny.loom' if MODO == "ejemplo" else 'scenic_data/expr_bcells.loom'
if MODO == "zenodo":
    row_attrs = {'Gene': np.array(ex_matrix.columns)}
    col_attrs = {'CellID': np.array(ex_matrix.index)}
    loompy.create(LOOM, ex_matrix.T.values, row_attrs, col_attrs)
    print("Loom de celulas B creado:", LOOM)

!pyscenic ctx scenic_data/adjacencies.tsv scenic_data/hg38_rankings.feather --annotations_fname scenic_data/motifs.tbl --expression_mtx_fname {LOOM} --output scenic_data/regulons.csv --num_workers 2

try:
    df = pd.read_csv('scenic_data/regulons.csv', index_col=[0,1], header=[0,1])
    n_regulons = len(df)
except Exception:
    n_regulons = 0
print(f"\nRegulones cisTarget: {n_regulons}")


### 4.5 · AUCell — actividad de regulones por célula

Con fallback si cisTarget no encuentra regulones (típico del dataset `tiny`).


In [ ]:
from pyscenic.aucell import aucell as pyscenic_aucell
from ctxcore.genesig import GeneSignature
import ast

signatures = []
if n_regulons > 0:
    print("Regulones de cisTarget")
    dfc = pd.read_csv('scenic_data/regulons.csv', header=[0,1], index_col=[0,1])
    dfc.columns = [' '.join(c).strip() for c in dfc.columns]; dfc = dfc.reset_index()
    tgt = next((c for c in dfc.columns if 'TargetGenes' in c), None)
    tfc = next((c for c in dfc.columns if c in ('TF','level_0')), None)
    for _, row in dfc.iterrows():
        try: genes = [t[0] for t in ast.literal_eval(str(row[tgt]))]
        except Exception: genes = []
        if genes: signatures.append(GeneSignature(name=f"{row[tfc]}(+)", gene2weight={g:1.0 for g in genes}))
else:
    print("Fallback: regulones desde el GRN (sin validacion de motivos)")
    for tf, grp in adjacencies.groupby('TF'):
        signatures.append(GeneSignature(name=f"{tf}(+)", gene2weight=dict(zip(grp['target'], grp['importance']))))

print(f"Firmas: {len(signatures)}")
auc_matrix = pyscenic_aucell(ex_matrix, signatures, num_workers=1)
print(f"AUCell: {auc_matrix.shape[0]} celulas x {auc_matrix.shape[1]} regulones")
auc_matrix.head()


### 4.6 · Visualizar la actividad de regulones

In [ ]:
import scanpy as sc
asc = sc.AnnData(X=auc_matrix.values,
                 obs=pd.DataFrame(index=auc_matrix.index.astype(str)),
                 var=pd.DataFrame(index=auc_matrix.columns.astype(str)))
sc.pp.neighbors(asc, random_state=42)
sc.tl.umap(asc, random_state=42)
sc.tl.leiden(asc, flavor='igraph', n_iterations=2, random_state=42)
sc.pl.umap(asc, color='leiden', title=f'Regulones activos ({MODO})')

out = f"{PROJECT_DIR}/scenic_auc_{MODO}.csv"
auc_matrix.to_csv(out); print("Guardado:", out)


---
# 5 · Resumen

Según el `MODO` que elegiste, este notebook corrió de principio a fin:

**MODO `"ejemplo"`** — clustering de médula ósea (UMAP + tipos celulares + genes diferenciales) y SCENIC sobre el dataset `tiny`.

**MODO `"zenodo"`** — datos reales de lupus: conversión R→Python con ahorro de RAM, exploración SLE vs control, Figura 1 (células B), Figura 2 (volcano pre/post-rituximab) y SCENIC sobre las células B reales.

**Parámetros clave** (celda 1): `MODO`, `N_CELLS_MAX`, `SCENIC_DOWNSAMPLE`, `SCENIC_N_CELLS`, `SCENIC_N_GENES`.

**Siguiente fase:** CloudPred — predicción clínica (SLE vs sano) a nivel de paciente.
